[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB-X1_box_prompt_segmentation.ipynb)

# NB-X1 · Challenge: segmentation with a box prompt (GrabCut, optional MedSAM)
**Days 4–5 · optional · about 40 minutes · for fast students.** A *prompted* model needs a hint (a box or a point) instead of being trained for one organ. We start with a classical prompted method that runs anywhere (GrabCut, OpenCV), then optionally compare it with a foundation model.

Needs `data/montgomery_256.npz` from Moodle (same file as NB08), or runs on a synthetic image if it is missing.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, cv2
from pathlib import Path
SEED = 0; rng = np.random.default_rng(SEED)
NPZ = Path("data/montgomery_256.npz")
if NPZ.exists():
    d = np.load(NPZ, allow_pickle=True); IMG_K = 0; i0 = int(d["test_idx"][IMG_K]); im = d["images"][i0].astype(np.float32)
    img = (im / im.max() * 255).astype(np.uint8) if im.max() > 1.5 else (im * 255).astype(np.uint8); gt = d["masks"][i0] > 0
    print("Montgomery test image", img.shape, "| keys:", list(d.keys()))
else:
    yy, xx = np.mgrid[:256, :256]; gt = ((yy - 128) ** 2 / 70 ** 2 + (xx - 128) ** 2 / 45 ** 2) < 1
    img = (np.where(gt, 150, 70) + rng.normal(0, 12, (256, 256))).clip(0, 255).astype(np.uint8); print("synthetic image (data file not found)")
def dice(a, b): a, b = a.astype(bool), b.astype(bool); return 2 * (a & b).sum() / max(a.sum() + b.sum(), 1)

## 1 · GrabCut with a box
GrabCut models foreground and background colours inside and outside a rectangle and refines the border. **TODO 1:** compute the tight bounding box of the ground-truth mask, widen it by `margin` pixels (a clinician's box is never perfect) and run GrabCut.

In [ ]:
def box_from_mask(m, margin=0):
    # TODO 1a: tight box (x0, y0, x1, y1) of the True pixels of m, widened by margin and clipped to the image
    return (0, 0, m.shape[1] - 1, m.shape[0] - 1)

def grabcut_box(image_gray, box, iters=5):
    # TODO 1b: run cv2.grabCut with cv2.GC_INIT_WITH_RECT; return a boolean foreground mask
    #   hint: image must be 3-channel (cv2.cvtColor), rect = (x, y, w, h), foreground = GC_FGD or GC_PR_FGD
    return np.zeros(image_gray.shape, bool)

box = box_from_mask(gt, margin=8); pred = grabcut_box(img, box)
print("box", box, "| Dice", round(dice(pred, gt), 3))

In [ ]:
x0, y0, x1, y1 = box; assert x1 > x0 and y1 > y0 and (x1 - x0) < 256
assert pred.shape == gt.shape and pred.dtype == bool and pred.any(), "GrabCut returned no foreground"
box_mask = np.zeros_like(gt); box_mask[y0:y1 + 1, x0:x1 + 1] = True
inv = box_mask & ~pred          # polarity check: the same split, but with foreground and background swapped inside the box
print(f"GrabCut Dice {dice(pred, gt):.3f} | swapped polarity (box minus GrabCut) {dice(inv, gt):.3f} | trivial baseline = the filled box: {dice(box_mask, gt):.3f}")
print(f"area as a share of the box: GrabCut {pred.sum() / box_mask.sum():.2f} | ground truth {gt.sum() / box_mask.sum():.2f}")
print("Checkpoint 1 passed (the mask exists; whether GrabCut beats the filled box is the QUESTION of this notebook, not an assumption)")
fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
ax[0].imshow(img, cmap="gray"); ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="y")); ax[0].set_title("image + box")
ax[1].imshow(pred, cmap="gray"); ax[1].set_title(f"GrabCut, Dice {dice(pred, gt):.2f}"); ax[2].imshow(gt, cmap="gray"); ax[2].set_title("ground truth")
for a in ax: a.axis("off")
plt.show()

**Dice exactly 0?** GrabCut only splits the box into two intensity groups; it does not know which one is "lung". On a chest X-ray the lungs are the *dark* region and the ribs, heart and mediastinum are bright, so GrabCut often labels the **wrong** side as foreground. The *swapped polarity* number above checks that: a high value means the split was right and the label was inverted (a classical failure of unsupervised methods that a learned model does not have).

**Look at the numbers.** On a chest X-ray the box around both lungs already contains most of the lung area, so the *filled box* is a strong baseline, and GrabCut (which models colour, not anatomy) often struggles on grey, low-contrast images. If GrabCut does **not** beat the box here, that is a result, not a bug. For comparison, the U-Net of NB08 reached test Dice about 0.97 on this data. **TODO (written):** one sentence on why a model that learned anatomy wins.

## 2 · How much does a sloppy box cost? (Plus)
Real users draw loose boxes. **TODO 2:** Dice for margins 0, 8, 16, 32 px and for a box **shifted** 15 px sideways. Report which direction hurts more.

In [ ]:
res = {}
# TODO 2: fill res[f"margin {m}"] = Dice for m in (0, 8, 16, 32), and res["shifted +15 px"] for a box moved 15 px to the right
# (if your Dice is 0, also try the swapped polarity: box_mask & ~pred, see the text above)
for k, v in res.items(): print(f"{k:16s} Dice {v:.3f}")
assert len(res) == 5, "expected 4 margins + 1 shifted box"

## 3 · Optional (Challenge): MedSAM
MedSAM is Segment Anything fine-tuned on medical images and takes the **same kind of box prompt**. It needs a ~375 MB checkpoint, a GPU, and a package that we did **not** pin or verify for this course. **Only try it if you have time**; follow the instructions in the official MedSAM repository (Ma et al., *Nature Communications* 2024) and compare Dice on the same box. Do not copy a checkpoint into the course repository.

**Reflection.** 1) GrabCut uses colour statistics, MedSAM uses learned anatomy: on a chest X-ray, where would each fail? 2) Who draws the box in a clinical workflow, and what does that do to the claim "automatic segmentation"? 3) A prompted model needs no training data for your organ. What do you give up?